In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

In [2]:
# =============================================================================
# ── 4. SILVER LAYER: Aggregation & Transformation (Commune level + Unpivot)
# =============================================================================
print("\n⚙️ Processing BRONZE -> SILVER...")

# Load Bronze
path_rhone_bronze = PROJECT_ROOT / "data" / "bronze" / "2022-pres-t1-commune-rhone-69-bronze.csv"

df_b = pd.read_csv(path_rhone_bronze, sep=";", dtype=str)

# --- A. Aggregate the Base Commune Metrics ---
base_cols = ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune']
metrics_cols = ['Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés']

# Convert metrics to integer for summing
for c in metrics_cols:
    df_b[c] = df_b[c].astype(int)


⚙️ Processing BRONZE -> SILVER...


In [3]:
# Group by Commune and sum the metrics
df_commune = df_b.groupby(base_cols)[metrics_cols].sum().reset_index()

# Add blancs_et_nuls column (sum for 2022 where they are separate)
df_commune['Blancs et nuls'] = df_commune['Blancs'] + df_commune['Nuls']

# Recalculate percentages correctly at the commune level
df_commune['% Abs/Ins'] = (df_commune['Abstentions'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Vot/Ins'] = (df_commune['Votants'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Ins'] = (df_commune['Blancs'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Blancs'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Nuls/Ins'] = (df_commune['Nuls'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Nuls/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Nuls'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Blancs et nuls/Ins'] = (df_commune['Blancs et nuls'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Blancs et nuls/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Blancs et nuls'] / df_commune['Votants'] * 100), 0).round(2)
df_commune['% Exp/Ins'] = (df_commune['Exprimés'] / df_commune['Inscrits'] * 100).round(2)
df_commune['% Exp/Vot'] = np.where(df_commune['Votants'] > 0, (df_commune['Exprimés'] / df_commune['Votants'] * 100), 0).round(2)


In [4]:
# --- B. Unpivot / Melt the Candidates Data ---
cand_dfs = []
for i in range(1, 13):
    # Extract columns for this specific candidate
    cand_subset = base_cols + [f'N°Panneau_{i}', f'Sexe_{i}', f'Nom_{i}', f'Prénom_{i}', f'Voix_{i}']
    temp = df_b[cand_subset].copy()
    
    # Standardize column names
    temp.rename(columns={
        f'N°Panneau_{i}': 'N°Panneau',
        f'Sexe_{i}': 'Sexe',
        f'Nom_{i}': 'Nom',
        f'Prénom_{i}': 'Prénom',
        f'Voix_{i}': 'Voix'
    }, inplace=True)
    cand_dfs.append(temp)



In [5]:
# Combine all candidates vertically
df_cands = pd.concat(cand_dfs, ignore_index=True)
df_cands['Voix'] = df_cands['Voix'].astype(int)

# Group by Commune AND Candidate to sum their votes
cand_group_cols = base_cols + ['N°Panneau', 'Sexe', 'Nom', 'Prénom']
df_cands_grouped = df_cands.groupby(cand_group_cols)['Voix'].sum().reset_index()

# --- C. Merge & Finalize Silver Dataset ---
# Join candidate votes with commune total metrics
df_silver = pd.merge(df_cands_grouped, df_commune, on=base_cols, how='left')

# Recalculate Candidate Percentages
df_silver['% Voix/Ins'] = (df_silver['Voix'] / df_silver['Inscrits'] * 100).round(2)
df_silver['% Voix/Exp'] = np.where(df_silver['Exprimés'] > 0, (df_silver['Voix'] / df_silver['Exprimés'] * 100), 0).round(2)



In [6]:
# Reorder columns for readability
final_cols = base_cols + metrics_cols + ['Blancs et nuls'] + [
    '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot',
    '% Nuls/Ins', '% Nuls/Vot', '% Blancs et nuls/Ins', '% Blancs et nuls/Vot',
    '% Exp/Ins', '% Exp/Vot',
    'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp'
]
df_silver = df_silver[final_cols]


In [7]:
# Save to Silver
path_rhone_silver = PROJECT_ROOT / "data" / "silver" / "2022-pres-t1-commune-rhone-69-silver.csv"
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")

print(f"✅ SILVER dataset created: {path_rhone_silver}")
print(f"📊 Unique Communes: {df_silver['Code de la commune'].nunique()}")
print(f"📊 Total Rows: {len(df_silver)} (Communes × 11 Candidates)")

# Show a clean preview for a single commune (e.g., 'Affoux')
print("\n🔍 Sample preview (Commune: Affoux):")
display(df_silver[df_silver['Libellé de la commune'] == 'Affoux'].head(5))

✅ SILVER dataset created: /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver/2022-pres-t1-commune-rhone-69-silver.csv
📊 Unique Communes: 267
📊 Total Rows: 3204 (Communes × 11 Candidates)

🔍 Sample preview (Commune: Affoux):


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,% Blancs et nuls/Vot,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp
0,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2.34,79.11,97.66,1,F,ARTHAUD,Nathalie,0,0.00,0.0
1,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2.34,79.11,97.66,10,F,PÉCRESSE,Valérie,17,5.38,6.8
2,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2.34,79.11,97.66,11,M,POUTOU,Philippe,0,0.00,0.0
3,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2.34,79.11,97.66,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4
4,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2.34,79.11,97.66,2,M,ROUSSEL,Fabien,4,1.27,1.6


In [8]:
def add_election_metadata(df, annee_election, tour):
    df = df.copy()

    # Ensure clean types
    df["annee_election"] = pd.to_numeric(annee_election, errors="coerce")
    df["tour"] = pd.to_numeric(tour, errors="coerce")

    return df

df_silver = add_election_metadata(df_silver, 2022, 1)

print(f"Colonnes finales: {list(df_silver.columns)}")
display(df_silver.head(5))

Colonnes finales: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', 'Blancs et nuls', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Blancs et nuls/Ins', '% Blancs et nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'annee_election', 'tour']


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour
0,69,Rhône,001,Affoux,316,60,256,4,2,250,...,97.66,1,F,ARTHAUD,Nathalie,0,0.00,0.0,2022,1
1,69,Rhône,001,Affoux,316,60,256,4,2,250,...,97.66,10,F,PÉCRESSE,Valérie,17,5.38,6.8,2022,1
2,69,Rhône,001,Affoux,316,60,256,4,2,250,...,97.66,11,M,POUTOU,Philippe,0,0.00,0.0,2022,1
3,69,Rhône,001,Affoux,316,60,256,4,2,250,...,97.66,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4,2022,1
4,69,Rhône,001,Affoux,316,60,256,4,2,250,...,97.66,2,M,ROUSSEL,Fabien,4,1.27,1.6,2022,1


In [9]:
def add_nuance_to_silver_data(df_silver, reference_file_path):
    import pandas as pd

    df_ref = pd.read_csv(reference_file_path, encoding="utf-8-sig").rename(columns={
        "Nom": "Prénom_ref",
        "Prénom": "Nom_ref"
    })

    for col in ["Nom", "Prénom"]:
        df_silver[col] = df_silver[col].astype(str).str.strip()

    for col in ["Nom_ref", "Prénom_ref"]:
        df_ref[col] = df_ref[col].astype(str).str.strip()

    df_ref = df_ref.drop_duplicates(subset=["Nom_ref", "Prénom_ref"])

    df_enriched = df_silver.merge(
        df_ref[["Nom_ref", "Prénom_ref", "nuance"]],
        left_on=["Nom", "Prénom"],
        right_on=["Nom_ref", "Prénom_ref"],
        how="left"
    ).drop(columns=["Nom_ref", "Prénom_ref"])

    return df_enriched

# Ajouter la nuance politique
reference_path = PROJECT_ROOT / "data" / "reference" / "nuance_politique_candidates_master.csv"
df_silver = add_nuance_to_silver_data(df_silver, reference_path)

# Vérifier que la colonne est bien présente
print(f"Colonnes finales: {list(df_silver.columns)}")
display(df_silver.head(5))

path_rhone_silver = PROJECT_ROOT / "data" / "silver" / "2022-pres-t1-commune-rhone-69-silver.csv"
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")

print(f"✅ Updated SILVER dataset created: {path_rhone_silver}")
print(f"📊 Columns: {list(df_silver.columns)}")
print(f"📊 Total Rows: {len(df_silver)}")
display(df_silver.head(5))

Colonnes finales: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', 'Blancs et nuls', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Blancs et nuls/Ins', '% Blancs et nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'annee_election', 'tour', 'nuance']


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour,nuance
0,69,Rhône,001,Affoux,316,60,256,4,2,250,...,1,F,ARTHAUD,Nathalie,0,0.00,0.0,2022,1,EXG
1,69,Rhône,001,Affoux,316,60,256,4,2,250,...,10,F,PÉCRESSE,Valérie,17,5.38,6.8,2022,1,LR
2,69,Rhône,001,Affoux,316,60,256,4,2,250,...,11,M,POUTOU,Philippe,0,0.00,0.0,2022,1,EXG
3,69,Rhône,001,Affoux,316,60,256,4,2,250,...,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4,2022,1,DLF
4,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2,M,ROUSSEL,Fabien,4,1.27,1.6,2022,1,COM


✅ Updated SILVER dataset created: /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver/2022-pres-t1-commune-rhone-69-silver.csv
📊 Columns: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimés', 'Blancs et nuls', '% Abs/Ins', '% Vot/Ins', '% Blancs/Ins', '% Blancs/Vot', '% Nuls/Ins', '% Nuls/Vot', '% Blancs et nuls/Ins', '% Blancs et nuls/Vot', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'annee_election', 'tour', 'nuance']
📊 Total Rows: 3204


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,...,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,annee_election,tour,nuance
0,69,Rhône,001,Affoux,316,60,256,4,2,250,...,1,F,ARTHAUD,Nathalie,0,0.00,0.0,2022,1,EXG
1,69,Rhône,001,Affoux,316,60,256,4,2,250,...,10,F,PÉCRESSE,Valérie,17,5.38,6.8,2022,1,LR
2,69,Rhône,001,Affoux,316,60,256,4,2,250,...,11,M,POUTOU,Philippe,0,0.00,0.0,2022,1,EXG
3,69,Rhône,001,Affoux,316,60,256,4,2,250,...,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4,2022,1,DLF
4,69,Rhône,001,Affoux,316,60,256,4,2,250,...,2,M,ROUSSEL,Fabien,4,1.27,1.6,2022,1,COM


In [10]:
import pandas as pd
from datetime import datetime

def standardize_common_columns(
    df,
    source_dataset,
    date_refresh=None
):
    df = df.copy()

    rename_map = {
        "Code du département": "code_departement",
        "Libellé du département": "libelle_departement",
        "Code de la commune": "code_commune",
        "Libellé de la commune": "libelle_commune",
        "Inscrits": "inscrits",
        "Abstentions": "abstentions",
        "Votants": "votants",
        "Blancs": "blancs",
        "Nuls": "nuls",
        "Blancs et nuls": "blancs_et_nuls",
        "Exprimés": "exprimes",
        "% Abs/Ins": "pct_abs_ins",
        "% Vot/Ins": "pct_vot_ins",
        "% Blancs/Ins": "pct_blancs_ins",
        "% Blancs/Vot": "pct_blancs_vot",
        "% Nuls/Ins": "pct_nuls_ins",
        "% Nuls/Vot": "pct_nuls_vot",
        "% Blancs et nuls/Ins": "pct_blancs_et_nuls_ins",
        "% Blancs et nuls/Vot": "pct_blancs_et_nuls_vot",
        "% Exp/Ins": "pct_exprimes_ins",
        "% Exp/Vot": "pct_exprimes_vot",
        "N°Panneau": "numero_panneau",
        "Sexe": "sexe",
        "Nom": "nom",
        "Prénom": "prenom",
        "Voix": "voix",
        "% Voix/Ins": "pct_voix_ins",
        "% Voix/Exp": "pct_voix_exprimes",
        "annee_election": "annee_election",
        "tour": "tour",
        "nuance": "nuance"
    }

    df = df.rename(columns=rename_map)

    df["code_departement"] = (
        df["code_departement"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(2)
    )

    df["code_commune"] = (
        df["code_commune"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(3)
    )

    df["libelle_departement"] = df["libelle_departement"].astype(str).str.strip()
    df["libelle_departement"] = df["libelle_departement"].replace({
    "Rhône": "RHONE",
    "Rhone": "RHONE",
    "RHÔNE": "RHONE",
    "rhône": "RHONE",
    "rhone": "RHONE",
})

    df["libelle_commune"] = df["libelle_commune"].astype(str).str.strip()
    df["nom"] = df["nom"].astype(str).str.strip()
    df["prenom"] = df["prenom"].astype(str).str.strip()
    df["nuance"] = df["nuance"].astype(str).str.strip()

    df["annee_election"] = pd.to_numeric(df["annee_election"], errors="coerce").astype("Int64")
    df["tour"] = pd.to_numeric(df["tour"], errors="coerce").astype("Int64")
    df["numero_panneau"] = pd.to_numeric(df["numero_panneau"], errors="coerce").astype("Int64")

    int_cols = [
        "inscrits", "abstentions", "votants", "blancs",
        "nuls", "blancs_et_nuls", "exprimes", "voix"
    ]
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    float_cols = [
        "pct_abs_ins", "pct_vot_ins", "pct_blancs_ins", "pct_blancs_vot",
        "pct_nuls_ins", "pct_nuls_vot", "pct_blancs_et_nuls_ins", "pct_blancs_et_nuls_vot",
        "pct_exprimes_ins", "pct_exprimes_vot",
        "pct_voix_ins", "pct_voix_exprimes"
    ]
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    

    df["source_dataset"] = source_dataset
    df["date_refresh"] = date_refresh or datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    ordered_cols = [
        "code_departement",
        "libelle_departement",
        "code_commune",
        "libelle_commune",
        "annee_election",
        "tour",
        "inscrits",
        "abstentions",
        "votants",
        "blancs",
        "nuls",
        "blancs_et_nuls",
        "exprimes",
        "pct_abs_ins",
        "pct_vot_ins",
        "pct_blancs_ins",
        "pct_blancs_vot",
        "pct_nuls_ins",
        "pct_nuls_vot",
        "pct_blancs_et_nuls_ins",
        "pct_blancs_et_nuls_vot",
        "pct_exprimes_ins",
        "pct_exprimes_vot",
        "numero_panneau",
        "sexe",
        "nom",
        "prenom",
        "voix",
        "pct_voix_ins",
        "pct_voix_exprimes",
        "nuance",
        "source_dataset",
        "date_refresh"
    ]
    return df[ordered_cols]


df_silver = standardize_common_columns(
    df_silver,
    source_dataset="2022-pres-t1-commune-rhone-69-silver.csv"
)


In [11]:
path_rhone_silver = PROJECT_ROOT / "data" / "silver" / "2022-pres-t1-commune-rhone-69-silver.csv"
df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")
print(f"✅ Saved: {path_rhone_silver}")

✅ Saved: /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver/2022-pres-t1-commune-rhone-69-silver.csv


In [12]:
import csv

# ✅ VERIFICATION: Check output schema standardization
print("\n" + "="*120)
print("📋 SCHEMA VERIFICATION: Output CSV Headers")
print("="*120 + "\n")

headers = pd.read_csv(path_rhone_silver, sep=";", nrows=0).columns.tolist()
print(f"✅ File: {path_rhone_silver}")
print(f"✅ Column count: {len(headers)}")
print(f"✅ Columns: {headers}\n")

# Check standardization
expected_lowercase_cols = [
    'code_departement', 'libelle_departement', 'code_commune', 'libelle_commune',
    'annee_election', 'tour', 'inscrits', 'abstentions', 'votants', 'blancs', 'nuls',
    'blancs_et_nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins',
    'pct_blancs_vot', 'pct_nuls_ins', 'pct_nuls_vot', 'pct_blancs_et_nuls_ins',
    'pct_blancs_et_nuls_vot', 'pct_exprimes_ins', 'pct_exprimes_vot',
    'numero_panneau', 'sexe', 'nom', 'prenom', 'voix', 'pct_voix_ins',
    'pct_voix_exprimes', 'nuance', 'source_dataset', 'date_refresh'
]

is_standardized = all(col in headers for col in expected_lowercase_cols)
if is_standardized:
    print("✅ SCHEMA IS FULLY STANDARDIZED (lowercase)")
    print(f"✅ All {len(expected_lowercase_cols)} expected columns present")
else:
    missing = [col for col in expected_lowercase_cols if col not in headers]
    print(f"❌ SCHEMA NOT FULLY STANDARDIZED")
    print(f"Missing columns: {missing}")

print("="*120 + "\n")


📋 SCHEMA VERIFICATION: Output CSV Headers

✅ File: /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver/2022-pres-t1-commune-rhone-69-silver.csv
✅ Column count: 33
✅ Columns: ['code_departement', 'libelle_departement', 'code_commune', 'libelle_commune', 'annee_election', 'tour', 'inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'blancs_et_nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_blancs_vot', 'pct_nuls_ins', 'pct_nuls_vot', 'pct_blancs_et_nuls_ins', 'pct_blancs_et_nuls_vot', 'pct_exprimes_ins', 'pct_exprimes_vot', 'numero_panneau', 'sexe', 'nom', 'prenom', 'voix', 'pct_voix_ins', 'pct_voix_exprimes', 'nuance', 'source_dataset', 'date_refresh']

✅ SCHEMA IS FULLY STANDARDIZED (lowercase)
✅ All 33 expected columns present

